# Train BASELINE MF — ML-10M (lưu weight lên Drive)

Baseline **MF** — model học thật, cá nhân hóa, để so sánh với DIVRS (giống paper).
MF nhẹ (sampler đều, không pinverse) → **CHẠY CPU ĐƯỢC** khi hết GPU. Weight ghi thẳng vào **Drive** mỗi epoch.

💡 Hết GPU Colab? **MF chạy CPU vẫn ổn** (giảm epoch). Hoặc dùng **Kaggle** (GPU free 30h/tuần, quota riêng).

## 1. GPU/CPU + lib

In [ ]:
!nvidia-smi || echo 'KHONG co GPU -> chay CPU (MF van kham duoc)'
!pip install -q absl-py setproctitle deprecated faiss-cpu
import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Code + data + Drive

In [ ]:
import os
REPO='/content/DIVRS'
if os.path.exists(REPO):
    !cd {REPO} && git fetch -q origin && git reset --hard -q origin/main
else:
    !git clone -q https://github.com/HatDuaa/DIVRS-reproduce.git {REPO}
%cd {REPO}
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/ml10m/output/train_coo_record.npz'):
    !cd data && wget -q https://raw.githubusercontent.com/tsinghua-fib-lab/DICE/main/data/ml10m.zip && unzip -oq ml10m.zip
from google.colab import drive; drive.mount('/content/drive')
OUT_DIR='/content/drive/MyDrive/Colab Notebooks/DIVRS_output'
os.makedirs(OUT_DIR, exist_ok=True)
print('Luu vao:', OUT_DIR)

## 3. Train MF + Test
`epochs=15` (đủ cho baseline; CPU thì còn nhẹ hơn). `es_patience=5` tự dừng sớm.
Checkpoint ghi mỗi epoch lên Drive → chạy được tới đâu cũng có weight dùng (kể cả dừng sớm).

In [ ]:
USE_GPU=torch.cuda.is_available(); print('use_gpu =', USE_GPU, '(False = dang chay CPU)')
!python app.py \
  --flagfile config/ml10m_mf.cfg \
  --output "{OUT_DIR}/" \
  --use_gpu={USE_GPU} --gpu_id=0 \
  --cg_use_gpu=False \
  --num_workers=2 \
  --epochs=15 --es_patience=5

## 4. Xem kết quả + checkpoint (đọc thẳng từ Drive)

In [ ]:
import glob
run=max([r for r in glob.glob(OUT_DIR+'/*/') if 'divrs' not in r.lower()], key=os.path.getmtime)
print('Run MF:', run)
print('===== TEST metrics =====')
!find "{run}test_log" -type f -exec grep -hE "TEST results|recall|hit_ratio|ndcg" {{}} + 2>/dev/null
print('===== Checkpoints tren Drive =====')
!ls -lah "{run}ckpt/" 2>/dev/null

## Ghi chú
- **CPU chậm nhưng được** với MF. Nhìn `it/s` ở epoch 1: nếu quá lâu (>10 phút/epoch) thì bấm ⏹ dừng — checkpoint các epoch đã xong vẫn nằm trên Drive, demo dùng được.
- **Kaggle** (kaggle.com/code → New Notebook → Settings → Accelerator GPU): GPU free 30h/tuần, quota riêng Colab. Code clone y hệt, chỉ cần upload data hoặc wget.
- Xong, mở `demo_DIVRS.ipynb` → **tự phát hiện run MF** làm baseline (thay Most-Popular).